In [6]:
import pypsa
import pandas as pd
import numpy as np

In [2]:
n_base = pypsa.Network('base_s_1__none_2035_lt.nc')
n_lt = n_base.copy()

INFO:pypsa.io:Imported network base_s_1__none_2035_lt.nc has buses, carriers, generators, global_constraints, links, loads, storage_units, stores


In [31]:
horizon = 48
overlap = 0

# overlap 8 funktioniert super

nr_networks = 2

co2 preise adden:
- einmal ohne biomass: specific_emissions
- einmal mit biomass: specific_emissions_biomass

In [20]:
from typing import Dict, Any

co2_prices = {
    "2020": 28,
    "2025": 70,
    "2030": 130,
    "2035": 190,
    "2040": 210,
    "2045": 530,
}

specific_emissions = {
    "oil" : 0.2571,
    "oil primary" : 0.2571,
    "gas" : 0.198, # OCGT
    "gas primary" : 0.198, # OCGT
    "coal" : 0.3361,
    "lignite" : 0.4069,
}

specific_emissions_biomass = {
    "oil" : 0.2571,
    "oil primary" : 0.2571,
    "gas" : 0.198, # OCGT
    "gas primary" : 0.198, # OCGT
    "coal" : 0.3361,
    "lignite" : 0.4069,
    "solid biomass": 0.4
}

def add_carbon_price(n, co2_price, specific_emissions):
        
    for carrier in specific_emissions.keys():
        n.generators.loc[n.generators.carrier == carrier, "marginal_cost"] += (
            co2_price * specific_emissions[carrier]
        )

def build_st_network(n, e_initial_h2 = 1e6):

    # alle hydrogen related stores
    stores = n.stores[n.stores.carrier.str.contains("H2", case=False, na=False)].index

    # marginal price für alle der mean
    for i in stores:
        n.stores.loc[i, 'marginal_cost'] = n_base.buses_t.marginal_price['DE0 0 H2'].mean()
        n.stores.loc[i, 'e_cyclic'] = False


    n.stores.loc['DE0 0 H2 Store-2035', 'e_initial'] = e_initial_h2
    n.stores.loc['DE0 0 H2 Store-2030', 'e_initial'] = 0
    n.stores.loc['DE0 0 H2 Store-2020', 'e_initial'] = 0

    n.optimize.fix_optimal_capacities()
    n.optimize.create_model()

    return n

In [21]:
onwind_cf = pd.read_pickle('Wetterdaten/cappacity_factors_prepared/cf_onwind.pkl')
offwind_cf = pd.read_pickle('Wetterdaten/cappacity_factors_prepared/cf_offwind.pkl')
solar_cf = pd.read_pickle('Wetterdaten/cappacity_factors_prepared/cf_solar.pkl')

In [22]:
np.random.seed(1)

choice = np.random.choice(np.arange(0, 50), nr_networks, replace=False)

choice


array([27, 35, 40, 38,  2,  3])

In [23]:

def change_onwind_cf(n,i):
    
    onwind_gen = n.generators[n.generators.carrier.str.contains("onwind", case=False, na=False)].index
    for g in onwind_gen:
        n.generators_t.p_max_pu[g] = onwind_cf[choice[i]]


def change_offwind_cf(n, i):
    offwind_gen = n.generators[n.generators.carrier.str.contains("offwind", case=False, na=False)].index
    for g in offwind_gen:
        n.generators_t.p_max_pu[g] = offwind_cf[choice[i]]


def change_solar_cf(n, i):
    n.generators_t.p_max_pu['DE0 0 solar-2035'] = solar_cf[choice[i]]
    n.generators_t.p_max_pu['DE0 0 solar-2020'] = solar_cf[choice[i]]
    n.generators_t.p_max_pu['DE0 0 solar-2005'] = solar_cf[choice[i]]
    n.generators_t.p_max_pu['DE0 0 solar-2015'] = solar_cf[choice[i]]
    n.generators_t.p_max_pu['DE0 0 solar-2010'] = solar_cf[choice[i]]


historic wind data mit biomass und rolling horizon

In [24]:
n_rh_with_biomass = n_lt.copy()
add_carbon_price(n_rh_with_biomass, co2_prices["2035"], specific_emissions_biomass)
build_st_network(n_rh_with_biomass)

n_rh_with_biomass.optimize.optimize_with_rolling_horizon(
    horizon = horizon,
    overlap = 0,
    solver_name='gurobi'
)

{'EU', 'DE'}
INFO:pypsa.optimization.abstract:Optimizing network for snapshot horizon [2019-01-01 00:00:00:2019-01-06 21:00:00] (1/61).
{'EU', 'DE'}
{'EU', 'DE'}
INFO:linopy.model: Solve problem using Gurobi solver


Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2637156


INFO:gurobipy:Set parameter LicenseID to value 2637156


...

INFO:gurobipy:


Solved in 1263 iterations and 0.12 seconds (0.04 work units)


INFO:gurobipy:Solved in 1263 iterations and 0.12 seconds (0.04 work units)


Optimal objective  2.103966329e+09


INFO:gurobipy:Optimal objective  2.103966329e+09
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 10040 primals, 20681 duals
Objective: 2.10e+09
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.


historic wind data ohne biomass und rolling horizon

In [28]:
n_rh_without_biomass = n_lt.copy()
add_carbon_price(n_rh_without_biomass, co2_prices["2035"], specific_emissions)
build_st_network(n_rh_without_biomass)

n_rh_without_biomass.optimize.optimize_with_rolling_horizon(
    horizon = horizon,
    overlap = 0,
    solver_name='gurobi'
)

{'EU', 'DE'}
INFO:pypsa.optimization.abstract:Optimizing network for snapshot horizon [2019-01-01 00:00:00:2019-01-06 21:00:00] (1/61).
{'EU', 'DE'}
{'EU', 'DE'}
INFO:linopy.model: Solve problem using Gurobi solver


Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2637156


INFO:gurobipy:Set parameter LicenseID to value 2637156


...

INFO:gurobipy:


Solved in 1340 iterations and 0.16 seconds (0.04 work units)


INFO:gurobipy:Solved in 1340 iterations and 0.16 seconds (0.04 work units)


Optimal objective  1.592436823e+09


INFO:gurobipy:Optimal objective  1.592436823e+09
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 10040 primals, 20681 duals
Objective: 1.59e+09
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper, Store-fix-e-lower, Store-fix-e-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance, Store-energy_balance were not assigned to the network.


Generation vergleich

In [29]:
gen = pd.DataFrame()
gen = pd.concat([gen, round(n_lt.statistics.supply(bus_carrier="AC").rename('N-LT') / 1e6)],axis=1)
gen = pd.concat([gen, round(n_rh_with_biomass.statistics.supply(bus_carrier="AC").rename('N-RH with Biomass') / 1e6)],axis=1)
gen = pd.concat([gen, round(n_rh_without_biomass.statistics.supply(bus_carrier="AC").rename('N-RH without Biomass') / 1e6)],axis=1) 
gen


N-LT  N-RH with Biomass  \
Generator   Offshore Wind (AC)                28.0               30.0   
            Offshore Wind (DC)               200.0              200.0   
            Onshore Wind                     266.0              279.0   
            Run of River                      15.0               15.0   
            Solar                            155.0              157.0   
            load-shedding                      0.0                1.0   
            solar-hsat                       165.0              166.0   
Link        CCGT                              26.0               25.0   
            H2 Fuel Cell                       0.0                0.0   
            H2 OCGT                            9.0                5.0   
            H2 retrofit OCGT                   1.0                NaN   
            Open-Cycle Gas                     0.0                0.0   
            battery discharger                53.0               57.0   
            coal                               3.0                2.0   
            lignite                            8.0                2.0   
            oil                                0.0                0.0   
            solid biomass                      0.0                0.0   
            urban central coal CHP             4.0                4.0   
            urban central gas CHP             41.0               47.0   
            urban central lignite CHP          0.0                0.0   
            urban central oil CHP              0.0                0.0   
            urban central solid biomass CHP    1.0                1.0   
StorageUnit Pumped Hydro Storage               4.0                4.0   
            Reservoir & Dam                    2.0                1.0   

                                             N-RH without Biomass  
Generator   Offshore Wind (AC)                               30.0  
            Offshore Wind (DC)                              199.0  
            Onshore Wind                                    278.0  
            Run of River                                     15.0  
            Solar                                           156.0  
            load-shedding                                     0.0  
            solar-hsat                                      164.0  
Link        CCGT                                             21.0  
            H2 Fuel Cell                                      0.0  
            H2 OCGT                                           2.0  
            H2 retrofit OCGT                                  NaN  
            Open-Cycle Gas                                    0.0  
            battery discharger                               57.0  
            coal                                              1.0  
            lignite                                           2.0  
            oil                                               0.0  
            solid biomass                                     0.0  
            urban central coal CHP                            2.0  
            urban central gas CHP                            25.0  
            urban central lignite CHP                         0.0  
            urban central oil CHP                             0.0  
            urban central solid biomass CHP                  18.0  
StorageUnit Pumped Hydro Storage                              4.0  
            Reservoir & Dam                                   1.0

Preisvergleich

In [30]:
lmps = pd.DataFrame()

lmps = pd.concat([lmps, n_lt.buses_t.marginal_price['DE0 0'].rename('N-LT')], axis=1)
lmps = pd.concat([lmps, n_rh_with_biomass.buses_t.marginal_price['DE0 0'].rename('N-RH with Biomass')], axis=1)
lmps = pd.concat([lmps, n_rh_without_biomass.buses_t.marginal_price['DE0 0'].rename('N-RH without Biomass')], axis=1)


lmps.describe()


,N-LT,N-RH with Biomass,N-RH without Biomass
count,2920.000000,2920.000000,2920.000000
mean,96.629048,114.024058,97.280168
std,127.512748,213.435382,213.524392
min,0.035115,0.035117,0.035117
25%,0.058193,14.728103,0.528063
50%,62.483105,86.356153,61.899084
75%,127.399611,125.076188,120.439399
max,1401.208942,2000.010004,2000.010004


In [ ]:
networks = []

for i in range(0, nr_networks):
    
    n = n_lt.copy()
    add_carbon_price(n, co2_prices["2035"], specific_emissions_biomass) # mit biomass

    change_onwind_cf(n, i)
    change_offwind_cf(n, i)
    change_solar_cf(n, i)

    n = build_st_network(n)

    n.optimize.optimize_with_rolling_horizon(
        horizon = horizon,
        overlap = overlap,
        solver_name='gurobi'
    )

    networks.append(n)

In [38]:
n_lt.buses_t.marginal_price['EU solid biomass']

snapshot
2019-01-01 00:00:00    76.529662
2019-01-01 03:00:00    76.529662
2019-01-01 06:00:00    76.529662
2019-01-01 09:00:00    76.529662
2019-01-01 12:00:00    76.529662
                         ...    
2019-12-31 09:00:00    76.529662
2019-12-31 12:00:00    76.529662
2019-12-31 15:00:00    76.529662
2019-12-31 18:00:00    76.529662
2019-12-31 21:00:00    76.529662
Name: EU solid biomass, Length: 2920, dtype: float64

In [39]:
n_rh_with_biomass.buses_t.marginal_price['EU solid biomass']

snapshot
2019-01-01 00:00:00    89.659376
2019-01-01 03:00:00    89.659376
2019-01-01 06:00:00    89.659376
2019-01-01 09:00:00    89.659376
2019-01-01 12:00:00    89.659376
                         ...    
2019-12-31 09:00:00    89.659376
2019-12-31 12:00:00    89.659376
2019-12-31 15:00:00    89.659376
2019-12-31 18:00:00    89.659376
2019-12-31 21:00:00    89.659376
Name: EU solid biomass, Length: 2920, dtype: float64